<a href="https://colab.research.google.com/github/byeonggyuk/nlp/blob/main/NLP_11%EB%B0%B0%ED%8F%AC%EC%9A%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11주차 · BERT 기반 문장 분류
**충북대학교 | 자연언어처리 | 임화정**

## 목표
BERT 입력과 토크나이저를 확인하고, 한국어 사전학습 모델을 이진 감성 분류에 파인튜닝합니다. 동일 분할의 TF-IDF/LR와 비교하고, validation으로 체크포인트를 선택한 뒤 test를 한 번 평가합니다.

## 실행 조건과 검증 범위
- 기본 경로 `RUN_BERT=False`: numpy·pandas·scikit-learn CPU baseline. 이 경로와 노트북 형식은 제작 과정에서 실행 검증했습니다.
- BERT 경로: Colab GPU 권장, 인터넷 및 모델 다운로드·저장 공간 필요. `klue/bert-base`는 사전학습 base이며 감성 분류층은 새로 초기화됩니다.
- **BERT 모델 다운로드·파인튜닝은 제작 환경에서 실행하지 않았습니다. BERT 성능/실행 결과를 제공하지 않습니다.** 아래 코드는 학습을 직접 수행하기 위한 완성 예제이며 실제 환경에서 패키지·자원 오류가 나면 오류 메시지를 확인하세요.
- 학생용 TODO를 완료한 다음 설정 셀의 RUN_BERT를 True로 바꾸고 처음부터 실행하세요. 모델/설정 확정 후에만 RUN_FINAL_TEST=True로 바꿉니다.
- 실행 도중 설정 변경 후 부분 재실행하면 상태가 섞일 수 있습니다. 새 학습은 커널을 재시작하세요. 최종 test를 본 뒤 같은 test로 튜닝하지 마세요.

Colab에서 별도 셀로 필요한 패키지를 설치하세요(설치 후 import 오류가 나면 런타임 재시작).
```python
%pip install torch transformers numpy pandas scikit-learn
```

말뭉치는 7주차와 같은 **자체작성60개 영화리뷰**입니다. 실무 데이터가 아니고 성능을 일반화할 수 없습니다.

**학생용 · TODO/응답란, BERT 직접 실행 필요**

In [22]:
import sys, re, json, random, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix
SEED = 42
RUN_BERT = True
RUN_FINAL_TEST = True
MODEL_NAME = 'klue/bert-base'
MAX_LENGTH = 64
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5
LABEL_NAMES = ['부정', '긍정']
SAVE_DIR = Path('week11_bert_checkpoint')
random.seed(SEED)
np.random.seed(SEED)
print('Python:', sys.version.split()[0], '| numpy:', np.__version__, '| pandas:', pd.__version__)
print('BERT 학습:', RUN_BERT, '| 최종평가:', RUN_FINAL_TEST)


Python: 3.13.18 | numpy: 2.4.6 | pandas: 2.2.3
BERT 학습: True | 최종평가: True


## 1. 데이터 정제·분할
공백 정리만 적용하고 부정어를 보존합니다. 동일 텍스트 상충 라벨과 중복을 검사합니다. 정확 중복 제거는 유사 리뷰/작성자 그룹 누수를 모두 방지하지 못합니다. 혼합 리뷰 라벨은 작성자가 의도한 전체 평가이며 다른 사람이 다르게 판단할 수 있습니다.

### 실습 A
train 60%, validation20%, test20%를 층화 분할하세요. 원문을 먼저 분할하며 test는 튜닝에 쓰지 않습니다.

In [13]:
positive_reviews = [
 '영화가 정말 재미있고 감동적이었다',
 '배우들의 연기가 훌륭해서 몰입했다',
 '스토리가 탄탄하고 결말이 좋았다',
 '영상이 아름답고 음악도 훌륭했다',
 '다시 보고 싶은 재미있는 영화다',
 '기대보다 훨씬 재미있어서 만족했다',
 '웃음과 감동을 함께 주는 좋은 작품이다',
 '연출이 섬세하고 배우의 연기가 자연스럽다',
 '시간이 아깝지 않은 훌륭한 영화다',
 '가족과 함께 즐겁게 볼 수 있었다',
 '주인공의 이야기가 마음에 오래 남았다',
 '긴장감이 넘치고 마지막까지 재미있었다',
 '지루할 틈 없이 흥미로운 장면이 이어졌다',
 '음악과 장면이 잘 어울려서 감동했다',
 '배우의 표정이 생생해서 마음이 움직였다',
 '친구에게 추천하고 싶은 멋진 작품이다',
 '결말이 따뜻해서 기분이 좋아졌다',
 '대사가 재치 있고 웃긴 장면이 많았다',
 '액션이 시원하고 전개가 빠르다',
 '오랜만에 만족스러운 영화를 만났다',
 '평이 나빴지만 나는 재미있게 봤다',
 '초반은 느렸지만 결말은 감동적이었다',
 '완벽하지는 않아도 충분히 볼 만했다',
 '유치할 줄 알았는데 생각보다 좋았다',
 '배우가 좋아서 봤는데 내용도 재미있었다',
 '재미없다는 말에 동의하지 않는다 나는 만족했다',
 '돈이 아깝지 않았다 다시 보고 싶다',
 '기대 안 했는데 정말 재미있다 ㅋㅋ',
 '크게 웃지는 않았지만 잔잔한 감동이 좋았다',
 '긴 영화인데도 지루하지 않고 몰입했다'
]
negative_reviews = [
 '영화가 너무 지루하고 재미없었다',
 '배우의 연기가 어색해서 몰입하기 어려웠다',
 '스토리가 엉성하고 결말이 실망스러웠다',
 '영상은 화려하지만 내용은 별로였다',
 '다시는 보고 싶지 않은 최악의 영화다',
 '기대했는데 너무 실망해서 화가 났다',
 '웃음도 감동도 없는 지루한 작품이다',
 '연출이 산만하고 배우의 연기가 부자연스럽다',
 '시간이 아까운 영화라 추천하지 않는다',
 '가족과 봤는데 모두 재미없다고 했다',
 '주인공의 이야기가 이해되지 않고 답답했다',
 '긴장감이 없고 마지막까지 지루했다',
 '흥미로운 장면이 없이 같은 내용만 반복한다',
 '음악이 너무 커서 대사에 집중하지 못했다',
 '배우의 표정이 과장되어서 보기 불편했다',
 '친구에게 추천하고 싶지 않은 작품이다',
 '결말이 허무해서 기분이 나빠졌다',
 '대사가 뻔하고 웃긴 장면도 없었다',
 '액션은 많지만 전개가 산만하다',
 '오랜만에 정말 실망스러운 영화를 만났다',
 '평이 좋았지만 나는 재미없게 봤다',
 '초반은 좋았지만 결말이 너무 별로였다',
 '배우는 훌륭하지만 영화 전체는 지루했다',
 '재미있을 줄 알았는데 생각보다 나빴다',
 '배우가 좋아서 봤는데 내용은 실망이었다',
 '재미있다는 말에 동의하지 않는다 나는 실망했다',
 '돈이 아까웠다 다시 보고 싶지 않다',
 '기대 많이 했는데 정말 재미없다 ㅠㅠ',
 '웃기는 장면도 있지만 전체적으로 불쾌했다',
 '짧은 영화인데도 지루해서 끝까지 못 봤다'
]
raw_df = pd.DataFrame({
    'text': positive_reviews + negative_reviews,
    'label': [1] * len(positive_reviews) + [0] * len(negative_reviews)
})

def normalize_text(text):
    return re.sub(r'\s+', ' ', str(text)).strip()

df = raw_df.dropna(subset=['text', 'label']).copy()
df['text'] = df['text'].map(normalize_text)
df = df[df['text'].str.len() > 0].copy()
# 실제 데이터는 같은 텍스트의 상충 라벨을 별도로 검토해야 한다.
assert df.groupby('text')['label'].nunique().max() == 1, '상충 라벨을 먼저 검토하세요.'
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
df['review_id'] = np.arange(len(df))
assert set(df['label']) == {0, 1}
print('정제 전/후:', len(raw_df), len(df))
display(df.groupby('label').size().rename(index={0:'부정', 1:'긍정'}).to_frame('count'))
display(df.sample(6, random_state=SEED))


정제 전/후: 60 60


,count
label,
부정,30
긍정,30


,text,label,review_id
0,영화가 정말 재미있고 감동적이었다,1,0
5,기대보다 훨씬 재미있어서 만족했다,1,5
36,웃음도 감동도 없는 지루한 작품이다,0,36
45,친구에게 추천하고 싶지 않은 작품이다,0,45
13,음악과 장면이 잘 어울려서 감동했다,1,13
54,배우가 좋아서 봤는데 내용은 실망이었다,0,54


In [14]:
train_df, heldout = train_test_split(df, test_size=0.4, stratify=df.label, random_state=SEED)
valid_df, test_df = train_test_split(heldout, test_size=0.5, stratify=heldout.label, random_state=SEED)
train_df, valid_df, test_df = [part.reset_index(drop=True) for part in (train_df, valid_df, test_df)]
for a,b in [(train_df,valid_df),(train_df,test_df),(valid_df,test_df)]:
    assert set(a.review_id).isdisjoint(set(b.review_id))
    assert set(a.text).isdisjoint(set(b.text))
display(pd.DataFrame({name: part.label.value_counts().sort_index() for name,part in
                     [('train',train_df),('validation',valid_df),('test',test_df)]}))


,train,validation,test
label,,,
0,18,6,6
1,18,6,6


## 2. TF-IDF + LR baseline
train에만 vectorizer를 fit합니다. 공백 unigram/bigram의 간단한 baseline입니다. 모든 BERT epoch는 같은 train/validation을 사용합니다. 최종 비교는 test에 대한 두 모델의 결과이며, test로 우승 모델을 골라 튜닝하는 용도로 쓰지 않습니다.

In [15]:
def metrics(y_true, y_pred):
    return {'accuracy': accuracy_score(y_true,y_pred),
            'precision_positive': precision_score(y_true,y_pred,pos_label=1,zero_division=0),
            'recall_positive': recall_score(y_true,y_pred,pos_label=1,zero_division=0),
            'f1_positive': f1_score(y_true,y_pred,pos_label=1,zero_division=0),
            'macro_f1': f1_score(y_true,y_pred,labels=[0,1],average='macro',zero_division=0)}
start = time.perf_counter()
baseline = Pipeline([
    ('tfidf',TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, ngram_range=(1,2))),
    ('clf',LogisticRegression(C=1.0,max_iter=1000,random_state=SEED))])
baseline.fit(train_df.text,train_df.label)
baseline_seconds = time.perf_counter()-start
baseline_valid = baseline.predict(valid_df.text)
display(pd.DataFrame([metrics(valid_df.label,baseline_valid)],index=['LR validation']))
print('baseline 학습 초:', baseline_seconds)
# test 예측은 아직 수행하지 않습니다.


,accuracy,precision_positive,recall_positive,f1_positive,macro_f1
LR validation,0.583333,0.571429,0.666667,0.615385,0.58042


baseline 학습 초: 0.045201884000107384


## 3. BERT·토크나이저 로드
`AutoTokenizer`와 모델은 같은 체크포인트를 사용합니다. 모델별 tokenizer 알고리즘은 다를 수 있으므로 모든 한국어 모델이 WordPiece라고 단정하지 않습니다. 여기서는 KLUE BERT의 실제 출력 토큰을 확인합니다.

새 classification head 초기화 경고는 정상입니다. 파인튜닝 전 랜덤 분류층의 예측을 감성 분석 결과로 해석하지 마세요. seed 고정은 모든 기기에서 완전 재현을 보장하지 않습니다.

### 실습 B
num_labels, id2label/label2id를 확인하고 MAX_LENGTH를 설명하세요. 학생용은 완료 확인 플래그를 바꿔야 BERT가 실행됩니다.

In [16]:
# TODO B: num_labels=2, 0=부정/1=긍정 규약과 MAX_LENGTH=64를 확인하고 아래 플래그를 True로 변경하세요.
BERT_TODO_COMPLETE = True
if RUN_BERT:
    if not BERT_TODO_COMPLETE:
        raise ValueError('학생용 BERT 실습 B를 확인한 뒤 완료 플래그를 True로 바꾸세요.')
    import torch, transformers
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,num_labels=2,id2label={0:'부정',1:'긍정'},label2id={'부정':0,'긍정':1})
    model.to(device)
    print('torch:', torch.__version__, '| transformers:', transformers.__version__, '| device:',device)
    example = tokenizer(train_df.text.iloc[0],truncation=True,max_length=MAX_LENGTH)
    print('실제 tokens:',tokenizer.convert_ids_to_tokens(example['input_ids']))
    print('attention_mask:',example['attention_mask'])
    print('token_type_ids:',example.get('token_type_ids','이 토크나이저가 생성하지 않음'))
    raw_lengths = [len(tokenizer(text,truncation=False)['input_ids']) for text in train_df.text]
    print('train 길이 최대:',max(raw_lengths),'| 잘리는 문장 수:',sum(n>MAX_LENGTH for n in raw_lengths))
else:
    print('BERT 미실행. TODO 완료 후 RUN_BERT=True로 설정하세요.')

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch: 2.14.1+cpu | transformers: 5.22.1 | device: cpu
실제 tokens: ['[CLS]', '재미있', '##을', '줄', '알', '##았', '##는데', '생각', '##보', '##다', '나빴', '##다', '[SEP]']
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
train 길이 최대: 14 | 잘리는 문장 수: 0


## 4. Dataset·동적 padding·평가 함수
batch별 가장 긴 길이에 padding합니다. `attention_mask`에서 padding은0입니다. 라벨은 long 정수이고 모델의 학습 입력에서 labels로 전달합니다. 불필요한 명사 추출이나 형태소 분석을 BERT 입력에 자동 적용하지 않습니다.

`model.train()`은 dropout 등을 학습 상태로, `eval()`은 평가 상태로 바꿉니다. `no_grad()`는 평가 시 기울기를 계산하지 않습니다.

In [17]:
if RUN_BERT:
    class ReviewDataset(Dataset):
        def __init__(self,frame): self.frame=frame.reset_index(drop=True)
        def __len__(self): return len(self.frame)
        def __getitem__(self,index):
            row=self.frame.iloc[index]
            return {'text':row.text,'label':int(row.label)}

    def collate_reviews(items):
        encoded=tokenizer([x['text'] for x in items],padding=True,truncation=True,
                          max_length=MAX_LENGTH,return_tensors='pt')
        encoded['labels']=torch.tensor([x['label'] for x in items],dtype=torch.long)
        return encoded

    train_loader=DataLoader(ReviewDataset(train_df),batch_size=BATCH_SIZE,shuffle=True,
        collate_fn=collate_reviews,generator=torch.Generator().manual_seed(SEED))
    valid_loader=DataLoader(ReviewDataset(valid_df),batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_reviews)

    def predict_frame(frame):
        loader=DataLoader(ReviewDataset(frame),batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_reviews)
        model.eval()
        logits_list=[]
        with torch.no_grad():
            for batch in loader:
                inputs={k:v.to(device) for k,v in batch.items() if k!='labels'}
                logits_list.append(model(**inputs).logits.cpu().numpy())
        logits=np.concatenate(logits_list,axis=0)
        return logits.argmax(axis=-1),logits
    batch=next(iter(valid_loader))
    print('batch shapes:',{k:tuple(v.shape) for k,v in batch.items()})
else:
    print('BERT Dataset과 평가 함수는 RUN_BERT=True일 때 정의됩니다.')


batch shapes: {'input_ids': (8, 17), 'token_type_ids': (8, 17), 'attention_mask': (8, 17), 'labels': (8,)}


## 5. 파인튜닝과 validation 체크포인트 선택
AdamW, lr=2e-5, epochs=3을 시작 설정으로 사용합니다. 보편적 최적값은 아닙니다. epoch마다 validation macro-F1을 계산하여 가장 높은 체크포인트를 저장하고 동률이면 먼저 나온 epoch를 유지합니다. **test는 사용하지 않습니다.**

### 실습 C
학생용 train_batch_step을 완성하세요. 필요한 순서는 optimizer.zero_grad → forward/loss → backward → gradient clipping → optimizer.step입니다. clipping은 선택한 안정화 수단입니다. loss가 줄어도 일반화가 보장되지 않습니다.

In [18]:
def train_batch_step(batch):
    batch = {k: v.to(device) for k, v in batch.items()}
    optimizer.zero_grad()
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    return loss.item()

if RUN_BERT:
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE)
    SAVE_DIR.mkdir(parents=True,exist_ok=True)
    best_f1=-1.0
    history=[]
    started=time.perf_counter()
    for epoch in range(1,EPOCHS+1):
        model.train()
        total_loss=0.0
        for batch in train_loader:
            total_loss += train_batch_step(batch)*len(batch['labels'])
        pred,_=predict_frame(valid_df)
        result=metrics(valid_df.label,pred)
        history.append({'epoch':epoch,'train_loss':total_loss/len(train_df),**result})
        if result['macro_f1']>best_f1:
            best_f1=result['macro_f1']
            best_epoch=epoch
            torch.save(model.state_dict(),SAVE_DIR/'best_state.pt')
        print(history[-1])
    bert_seconds=time.perf_counter()-started
    model.load_state_dict(torch.load(SAVE_DIR/'best_state.pt',map_location=device,weights_only=True))
    model.eval()
    model.config.save_pretrained(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)
    split_ids={name:part.review_id.tolist() for name,part in [('train',train_df),('validation',valid_df),('test',test_df)]}
    metadata={'model_name':MODEL_NAME,'seed':SEED,'max_length':MAX_LENGTH,'best_epoch':best_epoch,
              'learning_rate':LEARNING_RATE,'epochs':EPOCHS,'id2label':{0:'부정',1:'긍정'},
              'torch_version':torch.__version__,'transformers_version':transformers.__version__,
              'split_ids':split_ids,'data_source':'자체작성 교육용 영화리뷰60개'}
    (SAVE_DIR/'experiment.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
    display(pd.DataFrame(history))
    print('선택 epoch:',best_epoch,'| BERT 학습+validation 초:',bert_seconds)
else:
    print('BERT 미실행.TODO C 완료 후 실행하세요.')


{'epoch': 1, 'train_loss': 0.6678554481930203, 'accuracy': 0.9166666666666666, 'precision_positive': 1.0, 'recall_positive': 0.8333333333333334, 'f1_positive': 0.9090909090909091, 'macro_f1': 0.916083916083916}
{'epoch': 2, 'train_loss': 0.3837912148899502, 'accuracy': 0.9166666666666666, 'precision_positive': 1.0, 'recall_positive': 0.8333333333333334, 'f1_positive': 0.9090909090909091, 'macro_f1': 0.916083916083916}
{'epoch': 3, 'train_loss': 0.1781152014931043, 'accuracy': 1.0, 'precision_positive': 1.0, 'recall_positive': 1.0, 'f1_positive': 1.0, 'macro_f1': 1.0}


,epoch,train_loss,accuracy,precision_positive,recall_positive,f1_positive,macro_f1
0,1,0.667855,0.916667,1.0,0.833333,0.909091,0.916084
1,2,0.383791,0.916667,1.0,0.833333,0.909091,0.916084
2,3,0.178115,1.000000,1.0,1.000000,1.000000,1.000000


선택 epoch: 3 | BERT 학습+validation 초: 75.1284533019998


## 6. 저장·재로드 일치 검사
수업에서는 config/tokenizer와 state_dict를 저장합니다. Hugging Face의 save_pretrained 모델 가중치 파일과 같은 구조라고 가정하지 말고 아래처럼 config로 모델을 만든 뒤 state_dict를 로드하세요. 자신이 저장한 파일만 불러옵니다. Colab 저장소는 임시이므로 종료 전 체크포인트 폴더를 내려받거나 자신의 저장 공간에 보관하세요.

In [19]:
if RUN_BERT:
    from transformers import AutoConfig
    config=AutoConfig.from_pretrained(SAVE_DIR)
    reloaded=AutoModelForSequenceClassification.from_config(config).to(device)
    reloaded.load_state_dict(torch.load(SAVE_DIR/'best_state.pt',map_location=device,weights_only=True))
    reloaded.eval()
    tokenizer_reloaded=AutoTokenizer.from_pretrained(SAVE_DIR)
    sample_text=valid_df.text.iloc[0]
    sample=tokenizer_reloaded(sample_text,return_tensors='pt',truncation=True,max_length=MAX_LENGTH)
    sample={k:v.to(device) for k,v in sample.items()}
    with torch.no_grad():
        original=model(**sample).logits
        restored=reloaded(**sample).logits
    assert torch.allclose(original,restored,atol=1e-5)
    print('체크포인트 재로드 logits 일치')
    del reloaded
else:
    print('모델 미학습: 재로드 검사 미실행')


체크포인트 재로드 logits 일치


## 7. Validation 오류 분석
최종test 실행 전에 수행합니다. 부정어·반어·혼합감성·라벨모호성을 검토하고 test를 열지 않습니다.

### 실습 D
학생용 오류 필터를 수정하세요. BERT 결과는 실제 학습 후에만 추가됩니다.

In [20]:
valid_analysis=valid_df[['review_id','text','label']].copy()
valid_analysis['LR']=baseline_valid
if RUN_BERT:
    valid_analysis['BERT'],_=predict_frame(valid_df)
# TODO D: label과 LR 예측이 다른 행만 추출하세요.
errors=valid_analysis[valid_analysis['label'] != valid_analysis['LR']].copy()
display(errors)


,review_id,text,label,LR,BERT
4,50,평이 좋았지만 나는 재미없게 봤다,0,1,0
5,2,스토리가 탄탄하고 결말이 좋았다,1,0,1
6,56,돈이 아까웠다 다시 보고 싶지 않다,0,1,0
7,55,재미있다는 말에 동의하지 않는다 나는 실망했다,0,1,0
10,27,기대 안 했는데 정말 재미있다 ㅋㅋ,1,0,1


## 8. 최종 test 1회
모델/설정/체크포인트를 확정한 뒤 RUN_FINAL_TEST=True로 설정합니다. BERT가 학습되지 않으면 test는 열지 않습니다. kernel 재시작은 보호 플래그를 초기화하지만 test를 새 독립 데이터로 만들지 않습니다.

macro-F1은 클래스별 F1의 단순 평균입니다. precision/recall_positive는 긍정기준이며 부정 리뷰 탐지가 목적이면 부정 recall도 별도로 확인하세요. 성능 표의 학습시간은 LR fit vs BERT epochs+validation으로 범위가 다르며 다운로드시간을 제외하므로 엄밀한 자원 비교에는 범위를 통일하세요.

In [23]:
if not RUN_FINAL_TEST:
    print('test 봉인: 모델 확정 후 RUN_FINAL_TEST=True로 변경하세요.')
elif not RUN_BERT or 'best_epoch' not in globals():
    print('BERT 학습과 체크포인트 선택을 먼저 완료하세요. test 미실행.')
elif globals().get('TEST_EVALUATED',False):
    print('이미 최종 평가했습니다. 저장된 결과를 확인하세요.')
else:
    lr_test=baseline.predict(test_df.text)
    bert_test,bert_test_logits=predict_frame(test_df)
    results=pd.DataFrame([
        {'model':'LR',**metrics(test_df.label,lr_test),'training_seconds':baseline_seconds},
        {'model':'BERT',**metrics(test_df.label,bert_test),'training_seconds':bert_seconds}])
    TEST_EVALUATED=True
    display(results)
    print(classification_report(test_df.label,bert_test,labels=[0,1],target_names=LABEL_NAMES,zero_division=0))
    cm=confusion_matrix(test_df.label,bert_test,labels=[0,1])
    display(pd.DataFrame(cm,index=['실제부정','실제긍정'],columns=['예측부정','예측긍정']))
    test_analysis=test_df[['review_id','text','label']].copy()
    test_analysis['LR']=lr_test
    test_analysis['BERT']=bert_test
    display(test_analysis[test_analysis.label!=test_analysis.BERT])
    results.to_csv(SAVE_DIR/'test_metrics.csv',index=False,encoding='utf-8-sig')
    test_analysis.to_csv(SAVE_DIR/'test_predictions.csv',index=False,encoding='utf-8-sig')


,model,accuracy,precision_positive,recall_positive,f1_positive,macro_f1,training_seconds
0,LR,0.25,0.333333,0.5,0.4,0.2,0.045202
1,BERT,1.00,1.000000,1.0,1.0,1.0,75.128453


              precision    recall  f1-score   support

          부정       1.00      1.00      1.00         6
          긍정       1.00      1.00      1.00         6

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12



,예측부정,예측긍정
실제부정,6,0
실제긍정,0,6


,review_id,text,label,LR,BERT


## 9. 새 리뷰 추론
logits는 정규화 전 점수, softmax 출력은 이진 클래스 확률 형태의 값입니다. 확률 보정이 되어 있다고 가정하지 않습니다. 0.5 근처를 중립이라고 할 수 없고, 중립은 별도 학습하지 않았습니다. 상품 리뷰 예시는 라벨 없는 도메인 이동 탐색이지 정확도 평가가 아닙니다.

In [24]:
if RUN_BERT and 'best_epoch' in globals():
    new_texts=['연기는 좋았지만 결말은 너무 실망스러웠다',
               '기대 안 했는데 영화가 정말 재미있었다',
               '배송은 빠르지만 제품이 고장났다']
    model.eval()
    inputs=tokenizer(new_texts,return_tensors='pt',padding=True,truncation=True,max_length=MAX_LENGTH)
    with torch.no_grad():
        logits=model(**{k:v.to(device) for k,v in inputs.items()}).logits
        probabilities=torch.softmax(logits,dim=-1).cpu().numpy()
    display(pd.DataFrame({'text':new_texts,'prediction':[LABEL_NAMES[i] for i in probabilities.argmax(-1)],
                          'p_negative':probabilities[:,0],'p_positive':probabilities[:,1]}))
else:
    print('파인튜닝 모델이 없으므로 실제 BERT 예측을 제공하지 않습니다.')


,text,prediction,p_negative,p_positive
0,연기는 좋았지만 결말은 너무 실망스러웠다,부정,0.864133,0.135867
1,기대 안 했는데 영화가 정말 재미있었다,긍정,0.090060,0.909940
2,배송은 빠르지만 제품이 고장났다,부정,0.786760,0.213240


## 10. 오류 대응·보고서 과제
- CUDA 메모리 부족: batch_size를4 또는2로 줄이고 커널을 재시작합니다. MAX_LENGTH를 줄이면 잘림과 지표 영향을 점검해야 합니다.
- CUDA 없음: CPU로 가능하지만 느릴 수 있습니다. 설치/다운로드 실패는 인터넷·패키지 호환을 확인합니다.
- 새 분류층 초기화 경고: base 모델 로드 시 정상. 모든 가중치가 랜덤이라는 뜻은 아닙니다.
- loss만 하락하고 validation이 악화: epoch/학습률을 validation으로 검토합니다. 같은 작은validation을 많이 반복 튜닝하면 과적합할 수 있습니다.

### 실습·과제
A: 분할과 class 수를 설명하세요(코드는 예제로 제공).
B: 모델/라벨/길이 규약 확인 후 플래그를 수정하세요.
C: 학습 step을 완성하세요.
D: 오류 필터를 완성하세요.
1. tokenizer 실제 출력과 잘림 수, batch shape를 기록하세요.
2. epoch별 train loss/validation macro-F1과 선택epoch를 설명하세요.
3. test LR/BERT 지표와 혼동행렬을 비교하고 FP/FN을 해석하세요.
4. 오분류를 두 개 분석하세요(오류가 적으면 그 사실과 모호한 문장 분석을 기록).
5. BERT가 항상 우수하지 않은 이유와 시간/메모리 비용을 설명하세요.
6. 새 리뷰의 softmax확률을 해석하고 도메인 이동 한계를 쓰세요.

### 응답란
- 분할·라벨·전처리:
- 토큰/길이/마스크:
- validation과 선택epoch:
- test지표·FP/FN:
- 오류유형:
- 자원·도메인 한계:

### 제출 체크리스트
- [ ] TODO완료, 실제 BERT 학습 수행 여부 명시
- [ ] 시드/버전/모델/설정/데이터출처 기록
- [ ] checkpoint 선택은validation, test는최종1회
- [ ] 실행 결과와 체크포인트 보관, 확률을중립으로오해하지않음

제출안: `학번_이름_11주차.ipynb`와 README. 공식 일정·배점·제출 방식은 담당교원 안내를 따릅니다. 다음12주차에서는 문장 라벨 대신 토큰별 NER 라벨을 학습합니다.

- 분할·라벨·전처리: 자체 작성 영화리뷰 60개를 층화 분할해 train 36(18/18), validation 12(6/6), test 12(6/6)으로 나눴음. 라벨은 0=부정, 1=긍정
전처리는 공백 정리만 했고 부정어는 보존했음

- 토큰/길이/마스크: KLUE BERT tokenizer로 [CLS]와 [SEP]가 추가되고 하위 단어로 분리됨
attention_mask는 실제 토큰 위치가 1이고 padding 위치가 0
token_type_ids는 모두 0이다. train 최대 길이는 14토큰이고 잘린 문장은 0개

- validation과 선택 epoch: val macro-F1은 0.916 → 0.916 → 1.0, train loss는 0.668 → 0.384 → 0.178
epoch 3을 선택했고 LR의 validation macro-F1은 0.580

- test 지표·FP/FN: BERT는 accuracy와 macro-F1이 모두 1.0이고 FP 0, FN 0
LR은 accuracy 0.25, macro-F1 0.2이며 FP 6, FN 3 였음.

- 오류 유형: BERT는 오류가 없었음
LR은 부정어·반박("동의하지 않는다"), 혼합 감성("~지만"), 기대 역전("~할 줄 알았는데")에서 틀렸음

- 자원·도메인 한계: BERT는 LR 대비 학습 시간이 약 1,7배이고 모델이 약 445MB이었음
softmax는 보정되지 않았고 중립 클래스가 없음. 상품 리뷰는 도메인 이동 사례라 정확도를 평가할 수 없음

## 참고·원자료 보완
- 제공 「Part3 딥러닝 9–12주차 강의안」 11주차(15–18쪽)
- 제공 「06 텍스트 분류」 디자인
- [Hugging Face BERT 문서](https://huggingface.co/docs/transformers/model_doc/bert)
- [KLUE BERT 모델 카드](https://huggingface.co/klue/bert-base)

원자료의4문장 학습 예제를60문장 train/validation/test로확장했습니다. 별도검증 없는 학습 대신 validation checkpoint 선택 및test 봉인을 추가했습니다. API변화가잦은 Trainer 대신 PyTorch직접루프로핵심학습순서를보여줍니다. 대규모성능검증이나실무모델을제공하는자료가 아닙니다.